# Workshop 01 — SFT-LoRA on Qwen3-4B for Phishing Detection (Text)

**Run this in SageMaker Studio.** Total runtime: ~45–60 minutes (most is SageMaker provisioning + training).

## What you'll do

1. Pull a pre-built phishing dataset (1k phish + 1k benign) from HuggingFace.
2. Stage the dataset into your SageMaker AI Registry.
3. Launch a serverless **SFT-LoRA** fine-tuning job on **Qwen3-4B** (`huggingface-reasoning-qwen3-4b`).
4. Deploy the fine-tuned adapter as a real-time inference endpoint.
5. Test it against held-out examples — and optionally compare to the base model.

## What "serverless" means here

You don't pick an instance type, instance count, or fleet. SageMaker provisions the GPUs behind the scenes; you pay for the wallclock training time. The `SFTTrainer` API never asks for compute settings.

## Prerequisites (your IAM/setup)

- A SageMaker AI domain with Studio access
- The execution role has `AmazonSageMakerModelCustomizationCoreAccess` (or equivalent inline policy)
- An S3 bucket for training output
- You've accepted the Qwen3 EULA (we'll prompt below)

## 0. Install / upgrade SageMaker SDK

In [ ]:
%pip install -q --upgrade sagemaker datasets boto3 rich
# After this cell, **restart the kernel** to pick up the new SageMaker version.

## 1. Configuration — fill these in

Two things you must set:
1. `S3_OUTPUT_PATH` — your S3 bucket; SageMaker writes adapter weights and logs here
2. `ROLE_ARN` — your SageMaker execution role; if running in Studio, leave as `None` and we auto-detect

The HuggingFace dataset is already public at `gt2026workshop/phreshphish-2k`.

In [ ]:
HF_DATASET_ID = "gt2026workshop/phreshphish-2k"
HF_CONFIG_NAME = "text"
# S3 output — leave S3_BUCKET = None to use the account's DEFAULT SageMaker bucket
# (sagemaker-<region>-<account-id>, auto-created on first use). The session cell
# resolves S3_OUTPUT_PATH = s3://<bucket>/<prefix>/ once the session exists.
S3_BUCKET = None
S3_PREFIX = "qwen3-4b-phish-sft"
ROLE_ARN = None  # if None, we auto-detect from the Studio session

BASE_MODEL = "huggingface-reasoning-qwen3-4b"  # Qwen3-4B JumpStart ID
MODEL_PACKAGE_GROUP_NAME = "qwen3-4b-phish-sft"
EXPERIMENT_NAME = "qwen3-4b-phish-sft"
RUN_NAME = "workshop-run-01"

# ── MLflow ────────────────────────────────────────────────────
# Pass an EXISTING MLflow app ARN so you get live loss curves during training.
# If you leave this None, the SDK tries to auto-CREATE an MLflow app. When the
# account is already at its 3-app quota that create FAILS, and the SDK silently
# disables tracking — which is why you'd see NO training/validation loss curves.
# The next cell auto-discovers an existing app and fills this in when it's None.
MLFLOW_RESOURCE_ARN = None  # e.g. "arn:aws:sagemaker:us-east-1:<acct>:mlflow-app/<id>"

ACCEPT_EULA = True   # ⚠️ You are agreeing to the Qwen3 license terms

## 2. SageMaker session boilerplate

In [ ]:
import os, boto3
from sagemaker.core.helper.session_helper import Session
from rich import print as rprint

REGION = boto3.Session().region_name
sm_client = boto3.client("sagemaker", region_name=REGION)
sagemaker_session = Session(sagemaker_client=sm_client)

# Default SageMaker bucket (sagemaker-<region>-<acct>); override via S3_BUCKET above.
BUCKET = S3_BUCKET or sagemaker_session.default_bucket()
S3_OUTPUT_PATH = f"s3://{BUCKET}/{S3_PREFIX}/"


if ROLE_ARN is None:
    from sagemaker.core.helper.session_helper import get_execution_role
    ROLE_ARN = get_execution_role()


def discover_mlflow_app(session_client):
    """Return the ARN of an existing, ready MLflow app, or None.

    MLflow apps are an account-level resource capped at a quota (default 3).
    Reusing one avoids the 'ResourceLimitExceeded' you hit when the SDK tries
    to auto-create a fourth.
    """
    try:
        apps = []
        paginator = session_client.get_paginator("list_mlflow_apps")
        for page in paginator.paginate():
            apps.extend(page.get("Summaries", []))
    except Exception as e:
        print(f"Could not list MLflow apps ({e}). Pass MLFLOW_RESOURCE_ARN manually.")
        return None
    ready = [a for a in apps if a.get("Status") in ("Created", "Updated")]
    print(f"Found {len(apps)} MLflow app(s); {len(ready)} ready.")
    for a in apps:
        print(f"  - {a.get('Name')}  status={a.get('Status')}  arn={a.get('Arn')}")
    return ready[0]["Arn"] if ready else None


if MLFLOW_RESOURCE_ARN is None:
    MLFLOW_RESOURCE_ARN = discover_mlflow_app(sm_client)

if MLFLOW_RESOURCE_ARN:
    print(f"\nUsing MLflow app: {MLFLOW_RESOURCE_ARN}")
else:
    print("\n⚠️  No usable MLflow app found. Training will run but loss curves "
          "will NOT be tracked.\n   Either free a slot (delete an old app) or create one, "
          "then set MLFLOW_RESOURCE_ARN above.\n   List/delete:  aws sagemaker list-mlflow-apps  |  "
          "aws sagemaker delete-mlflow-app --mlflow-app-name <name>")

rprint({"region": REGION, "role": ROLE_ARN, "output": S3_OUTPUT_PATH, "mlflow": MLFLOW_RESOURCE_ARN})

## 3. Pull the prepared dataset and stage it for SageMaker

SageMaker's `DataSet.create()` accepts an S3 URI to a JSONL file. We:
1. Download `train.jsonl` and `validation.jsonl` from HF
2. Upload them to your S3 bucket
3. Register them with the AI Registry

In [ ]:
import json, pathlib
from datasets import load_dataset

ds = load_dataset(HF_DATASET_ID, name=HF_CONFIG_NAME)
print(ds)
print("Sample row:", ds["train"][0])

In [ ]:
tmp = pathlib.Path("./_staged")
tmp.mkdir(exist_ok=True)

for split in ["train", "validation"]:
    out = tmp / f"{split}.jsonl"
    with open(out, "w") as f:
        for r in ds[split]:
            # SageMaker SFTTrainer expects {"prompt": ..., "completion": ...} per line.
            f.write(json.dumps({"prompt": r["prompt"], "completion": r["completion"]}) + "\n")
    print(f"wrote {out}  rows={len(ds[split])}  size={out.stat().st_size:,} bytes")

In [ ]:
# Upload the JSONL files to S3 (under your S3_OUTPUT_PATH/data/)
from urllib.parse import urlparse

p = urlparse(S3_OUTPUT_PATH.rstrip("/"))
BUCKET = p.netloc
PREFIX = p.path.lstrip("/")
DATA_PREFIX = f"{PREFIX}/data"

s3 = boto3.client("s3")
TRAIN_S3 = f"s3://{BUCKET}/{DATA_PREFIX}/train.jsonl"
VAL_S3 = f"s3://{BUCKET}/{DATA_PREFIX}/validation.jsonl"

s3.upload_file(str(tmp / "train.jsonl"), BUCKET, f"{DATA_PREFIX}/train.jsonl")
s3.upload_file(str(tmp / "validation.jsonl"), BUCKET, f"{DATA_PREFIX}/validation.jsonl")

print(f"train: {TRAIN_S3}")
print(f"val:   {VAL_S3}")

In [ ]:
# Register both splits with the SageMaker AI Registry
from sagemaker.ai_registry.dataset import DataSet

train_dataset = DataSet.create(
    name="phreshphish-workshop-text-train",
    source=TRAIN_S3,
    wait=True,
)
val_dataset = DataSet.create(
    name="phreshphish-workshop-text-val",
    source=VAL_S3,
    wait=True,
)
rprint({"train_arn": train_dataset.arn, "val_arn": val_dataset.arn})

## 4. Create the model package group

This is the registry collection your fine-tuned adapter will land in.

In [ ]:
from sagemaker.core.resources import ModelPackageGroup

try:
    model_package_group = ModelPackageGroup.create(
        model_package_group_name=MODEL_PACKAGE_GROUP_NAME,
        model_package_group_description="Qwen3-4B fine-tuned for phishing detection — GovTech GASP workshop",
    )
    print("created", model_package_group.model_package_group_arn)
except Exception as e:
    # already exists — that's fine
    print("reusing existing group:", MODEL_PACKAGE_GROUP_NAME, "|", e)
    model_package_group = ModelPackageGroup.get(model_package_group_name=MODEL_PACKAGE_GROUP_NAME)

## 5. Build the SFTTrainer (LoRA)

Two things to notice:
- `training_type=TrainingType.LORA` — adapter training, not full fine-tune. Faster, cheaper, smaller artifact.
- No `instance_type` parameter. Compute is managed by the serverless customization service.

In [ ]:
from sagemaker.train.sft_trainer import SFTTrainer
from sagemaker.train.common import TrainingType

trainer = SFTTrainer(
    model=BASE_MODEL,
    training_type=TrainingType.LORA,
    model_package_group=model_package_group,
    training_dataset=train_dataset.arn,
    validation_dataset=val_dataset.arn,
    s3_output_path=S3_OUTPUT_PATH,
    sagemaker_session=sagemaker_session,
    accept_eula=ACCEPT_EULA,
    mlflow_resource_arn=MLFLOW_RESOURCE_ARN,   # ← reuse existing app; None => no curves
    mlflow_experiment_name=EXPERIMENT_NAME,
    mlflow_run_name=RUN_NAME,
)

### Inspect the default hyperparameters

Each (model, training_type) pair has its own defaults. We print them so you know what's on the table — and tweak `global_batch_size` or epochs as a demonstration.

In [ ]:
from rich.pretty import pprint
print("Default hyperparameters for Qwen3-4B + LoRA:")
pprint(trainer.hyperparameters.to_dict())

In [ ]:
# Optional tweaks — uncomment any you want. Names exposed depend on the model.
# trainer.hyperparameters.global_batch_size = 16
# trainer.hyperparameters.epochs = 2
# trainer.hyperparameters.learning_rate = 2e-4
# trainer.hyperparameters.lora_rank = 16
# trainer.hyperparameters.lora_alpha = 32

pprint(trainer.hyperparameters.to_dict())

## 6. Launch training

`wait=True` blocks the cell until completion and streams MLflow metrics live in Studio. Set `wait=False` to fire-and-forget.

In [ ]:
training_job = trainer.train(wait=True)
print("Training job:", training_job.training_job_name)
print("Status:      ", training_job.training_job_status)

### Where are the loss curves?

While the job runs (and after), open **MLflow** from SageMaker Studio:
*Studio → Applications → MLflow → open the app whose ARN is printed above → experiment `qwen3-4b-phish-sft` → run `workshop-run-01`.*

You'll see `train_loss` and `eval_loss` per step/epoch. If the curves are missing, the cause is almost always that `MLFLOW_RESOURCE_ARN` was `None` at launch (auto-create hit the account's MLflow-app quota and tracking was disabled). Fix the ARN and re-launch.

The same metrics are also written to TensorBoard under `s3://<bucket>/<prefix>/.../tensorboard/`.

## 7. (Optional) Quick SageMaker endpoint for a smoke test

For a fast in-session sanity check you can spin up a SageMaker real-time endpoint with `ModelBuilder`. This is **billed per hour while it runs** — fine for a 10-minute smoke test, then tear it down in §10.

**For the real deployment, use notebook `05_deploy_bedrock_custom_model_import.ipynb`** — it imports the model into Amazon Bedrock (serverless, pay-per-token, scales to zero) instead of holding a GPU endpoint. Skip this section entirely if you're going straight to Bedrock.

Formal accuracy evaluation lives in notebooks 03 (base) and 04 (fine-tuned).

In [ ]:
RUN_SMOKE_TEST = False  # set True to deploy a temporary SageMaker endpoint for a quick check

if RUN_SMOKE_TEST:
    import random
    from sagemaker.serve import ModelBuilder

    endpoint_name = f"qwen3-4b-phish-{random.randint(1000, 9999)}"
    print("Endpoint will be:", endpoint_name)
    model_builder = ModelBuilder(model=trainer)
    model_builder.build()
    endpoint = model_builder.deploy(endpoint_name=endpoint_name)
    print("Deployed:", endpoint_name)
else:
    print("Skipping SageMaker endpoint. Use notebook 05 for Bedrock deployment.")

## 8. (Optional) Smoke-test the endpoint

Only runs if `RUN_SMOKE_TEST = True` above. For real evaluation use notebooks 03/04.

In [ ]:
if not RUN_SMOKE_TEST:
    print("No endpoint deployed (RUN_SMOKE_TEST=False). Skipping. See notebooks 03/04 for evaluation, 05 for Bedrock.")
else:
    import json, boto3
    
    rt = boto3.client("sagemaker-runtime", region_name=REGION)
    
    def predict(prompt: str, max_new_tokens: int = 5) -> str:
        body = {"inputs": prompt, "parameters": {"max_new_tokens": max_new_tokens, "temperature": 0.0, "do_sample": False}}
        resp = rt.invoke_endpoint(EndpointName=endpoint_name, Body=json.dumps(body), ContentType="application/json")
        out = json.loads(resp["Body"].read())
        # Output schema depends on the served container; common shapes below.
        if isinstance(out, list) and out and "generated_text" in out[0]:
            return out[0]["generated_text"]
        if isinstance(out, dict) and "generated_text" in out:
            return out["generated_text"]
        return str(out)
    
    # Pull 20 held-out examples and score them
    samples = ds["validation"].select(range(20))
    correct = 0
    for r in samples:
        pred = predict(r["prompt"]).strip().split()[0].lower().strip(".'\"")
        truth = r["completion"]
        ok = pred == truth
        correct += ok
        print(f"{'✓' if ok else '✗'}  truth={truth:<7}  pred={pred:<10}")
    
    print(f"\naccuracy on first 20 val samples: {correct}/20 = {correct/20:.1%}")

## 9. (Optional) Compare to the base model

Deploy the un-fine-tuned Qwen3-4B and run the same eval to see the lift from SFT.

Skip this cell if you're tight on time — it adds ~10 minutes for the second deploy.

## 10. Clean up

Endpoints cost money while they run. Delete when you're done.

In [ ]:
# Uncomment to tear down the endpoint at the end of the workshop.
# sm_client.delete_endpoint(EndpointName=endpoint_name)
# sm_client.delete_endpoint_config(EndpointConfigName=endpoint_name)